# 23. Selection receipts: what layout was chosen, and why

`CalibrationMapper` picks physical qubits using live calibration data. This notebook shows the
**selection receipt**: a machine-readable record of the layout it chose, the score it gave, and the
per-signal breakdown behind that score.

**What this is:** a reproducible record of a decision the compiler already made.

**What this is not:** a new layout algorithm. The receipt is derived purely from the mapper's own
result, so it can never disagree with what actually ran.

The receipt is Apache-2.0 and has no paid dependency. Reading why a layout was chosen is free.
Signing and storing that record is what the paid layer adds, and it is off by default.


## Setup: a small 4-qubit device with deliberately uneven qubits

Qubit 0 is the best (T1 300us, readout 0.5%), qubit 3 the worst (T1 100us, readout 5%). The
couplings degrade the same way, from 0.2% error on (0,1) to 1.5% on (2,3). A calibration-aware
mapper should prefer the low end of the chain.


In [1]:
from qb_compiler.calibration.models.backend_properties import BackendProperties
from qb_compiler.calibration.models.coupling_properties import GateProperties
from qb_compiler.calibration.models.qubit_properties import QubitProperties
from qb_compiler.ir.circuit import QBCircuit
from qb_compiler.ir.operations import QBGate
from qb_compiler.passes.mapping import (
    CalibrationMapper,
    calibration_fingerprint,
    selection_receipt,
)

qubit_props = [
    QubitProperties(qubit_id=0, t1_us=300.0, t2_us=250.0, readout_error=0.005),
    QubitProperties(qubit_id=1, t1_us=200.0, t2_us=180.0, readout_error=0.010),
    QubitProperties(qubit_id=2, t1_us=150.0, t2_us=120.0, readout_error=0.030),
    QubitProperties(qubit_id=3, t1_us=100.0, t2_us=80.0,  readout_error=0.050),
]
gate_props = [
    GateProperties(gate_type='cz', qubits=(0, 1), error_rate=0.002, gate_time_ns=68.0),
    GateProperties(gate_type='cz', qubits=(1, 0), error_rate=0.002, gate_time_ns=68.0),
    GateProperties(gate_type='cz', qubits=(1, 2), error_rate=0.008, gate_time_ns=68.0),
    GateProperties(gate_type='cz', qubits=(2, 1), error_rate=0.008, gate_time_ns=68.0),
    GateProperties(gate_type='cz', qubits=(2, 3), error_rate=0.015, gate_time_ns=68.0),
    GateProperties(gate_type='cz', qubits=(3, 2), error_rate=0.015, gate_time_ns=68.0),
]

backend = BackendProperties(
    backend='demo_heron',
    provider='demo',
    n_qubits=4,
    basis_gates=('cz', 'rz', 'sx', 'x', 'id'),
    coupling_map=[(0, 1), (1, 0), (1, 2), (2, 1), (2, 3), (3, 2)],
    qubit_properties=qubit_props,
    gate_properties=gate_props,
    timestamp='2026-03-12T00:00:00',
)
print(f'{backend.backend}: {backend.n_qubits} qubits, basis {backend.basis_gates}')


demo_heron: 4 qubits, basis ('cz', 'rz', 'sx', 'x', 'id')


## Run the mapper on a Bell pair


In [2]:
circuit = QBCircuit(n_qubits=2, name='bell')
circuit.add_gate(QBGate(name='h', qubits=(0,)))
circuit.add_gate(QBGate(name='cx', qubits=(0, 1)))

mapper = CalibrationMapper(backend)
result = mapper.run(circuit, {})

print('chosen layout :', result.metadata['initial_layout'])
print('score         :', result.metadata['calibration_score'])


chosen layout : {0: 0, 1: 1}
score         : 0.14866666666666664


## The receipt

One call. It reads the mapper's result and records the decision.


In [3]:
receipt = selection_receipt(result, calibration=backend)

import json
print(json.dumps(receipt, indent=2, default=str))


{
  "schema": "qb.selection_receipt.v1",
  "objective": "calibration-aware layout (CalibrationMapper: gate error + coherence + readout + T1 asymmetry + temporal correlation, VF2 subgraph search)",
  "selected_layout": {
    "0": 0,
    "1": 1
  },
  "selected_score": 0.14866666666666664,
  "score_breakdown": {
    "gate_error": 0.02,
    "coherence": 0.05366666666666666,
    "readout": 0.07500000000000001,
    "t1_asymmetry": 0.0,
    "correlation": 0.0,
    "total": 0.14866666666666667
  },
  "calibration_hash": "b9072cd78a59d54b",
  "signature": null,
  "signing": "unsigned"
}


### Reading it

- `selected_layout` is the mapper's chosen virtual-to-physical mapping
- `selected_score` is the objective value it achieved
- `score_breakdown` decomposes that score per signal: gate error, coherence, readout, T1
  asymmetry, temporal correlation. This is the *why*.
- `calibration_hash` fingerprints the calibration snapshot used, so a receipt can be tied back to
  the device state at the time
- `signing` is `unsigned` here, which is the default


In [4]:
print('signals that drove the choice:')
for signal, value in (receipt['score_breakdown'] or {}).items():
    print(f'  {signal:<28} {value}')


signals that drove the choice:
  gate_error                   0.02
  coherence                    0.05366666666666666
  readout                      0.07500000000000001
  t1_asymmetry                 0.0
  correlation                  0.0
  total                        0.14866666666666667


## The receipt cannot drift from what ran

It is derived from the mapper's result rather than recomputed, so the layout in the receipt is
the layout that was actually used. Worth asserting, because a record that can disagree with
reality is worse than no record.


In [5]:
assert receipt['selected_layout'] == {
    str(k): v for k, v in result.metadata['initial_layout'].items()
}
assert receipt['selected_score'] == result.metadata['calibration_score']
print('receipt matches the mapper result exactly')


receipt matches the mapper result exactly


## Provenance: the calibration fingerprint

The same calibration snapshot always fingerprints the same way, and a different one does not.
That is what lets you tell whether two runs saw the same device state.


In [6]:
same = calibration_fingerprint(backend)

stale = BackendProperties(
    backend='demo_heron', provider='demo', n_qubits=4,
    basis_gates=('cz', 'rz', 'sx', 'x', 'id'),
    coupling_map=[(0, 1), (1, 0), (1, 2), (2, 1), (2, 3), (3, 2)],
    qubit_properties=qubit_props, gate_properties=gate_props,
    timestamp='2026-03-13T00:00:00',   # one day later
)

print('same snapshot   :', same, '==', calibration_fingerprint(backend))
print('later snapshot  :', calibration_fingerprint(stale))
print('differs         :', calibration_fingerprint(stale) != same)


same snapshot   : b9072cd78a59d54b == b9072cd78a59d54b
later snapshot  : c0abae2f8d9c6092
differs         : True


## Signing is optional, and absent by default

`sign=True` uses the QubitBoost SDK's Ed25519 signer **only if it is installed**. It is a soft
import: with no SDK present the receipt is still produced, unsigned, with a pointer. No hard
dependency, no nagging, and nothing breaks in a pure Apache install.

The cell below reports which path this environment took, so the output is unambiguous either
way. On a plain `pip install qb-compiler` you will see `unsigned`.


In [7]:
unsigned = selection_receipt(result)
print('default signing :', unsigned['signing'])
print('signature       :', unsigned['signature'])

# sign=True only does anything when the optional paid SDK is importable.
attempted = selection_receipt(result, calibration=backend, sign=True)
print()
print('with sign=True  :', attempted['signing'])

try:
    import qubitboost_sdk  # noqa: F401
    print('   -> the optional qubitboost-sdk IS installed in this environment, so it signed.')
    print('      On a plain `pip install qb-compiler` you will see "unsigned" here instead,')
    print('      and everything above still works.')
except ImportError:
    print('   -> the optional qubitboost-sdk is NOT installed, so the receipt stayed unsigned.')
    print('      That is the normal Apache-only path: nothing failed, nothing nagged.')


default signing : unsigned
signature       : None



with sign=True  : ed25519 (qubitboost_sdk)
   -> the optional qubitboost-sdk IS installed in this environment, so it signed.
      On a plain `pip install qb-compiler` you will see "unsigned" here instead,
      and everything above still works.


## Summary

| | |
|---|---|
| What you get free | the layout chosen, its score, the per-signal breakdown, a calibration fingerprint |
| What the paid layer adds | an Ed25519 signature and durable storage of that record |
| Dependency | none. Apache-2.0, signer soft-imported only if present |
| Guarantee | the receipt is derived from the mapper result, so it cannot disagree with what ran |
